(Instrução Direta ao Aluno) > Volte ao Colab. Crie um notebook Lab02_XGBoost. > Vamos prever quais clientes vão cancelar um serviço (Churn) usando dados massivos.
(Passo 1: Instalação do XGBoost para Spark) O XGBoost não vem no Spark padrão. Precisamos baixar o JAR.


In [1]:
# --- Célula 1: Setup com XGBoost ---
# Reiniciar Spark com o pacote do XGBoost
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

spark = SparkSession.builder \
.master("local[*]") \
.appName("Lab02_XGBoost") \
.config("spark.jars.packages", "ml.dmlc:xgboost4j-spark_2.12:1.7.6") \
.getOrCreate()

print("Spark com XGBoost configurado!")
# (Passo 2: Gerando Dados de Churn) Vamos simular um dataset de telecomunicações.


Spark com XGBoost configurado!


In [2]:
# --- Célula 2: Gerando Dados Fake ---
from pyspark.sql.functions import rand, when, col

# Criar 100 mil clientes
df = spark.range(0, 100000).select(
(rand(seed=42) * 100).alias("fatura_mensal"),
(rand(seed=23) * 10).alias("anos_contrato"),
(rand(seed=11) * 50).alias("idade"),
(rand(seed=99)).alias("score_satisfacao")
)

# Criar Label (Churn) com base em regras (para o modelo aprender algo)
# Se satisfação baixa e fatura alta -> Churn (1)
df = df.withColumn("churn",
when((col("score_satisfacao") < 0.3) & (col("fatura_mensal") > 80), 1.0)
.otherwise(0.0)
)

print("Distribuição do Target:")
df.groupBy("churn").count().show()
# (Passo 3: Pipeline com Random Forest (Nativo)) Primeiro, vamos usar o Random Forest nativo do Spark.


Distribuição do Target:
+-----+-----+
|churn|count|
+-----+-----+
|  0.0|93998|
|  1.0| 6002|
+-----+-----+



In [3]:
# --- Célula 3: Random Forest Nativo ---
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Assembler
assembler = VectorAssembler(
inputCols=["fatura_mensal", "anos_contrato", "idade", "score_satisfacao"],
outputCol="features"
)

# 2. Split
train, test = df.randomSplit([0.7, 0.3], seed=42)

# 3. Modelo
rf = RandomForestClassifier(labelCol="churn", featuresCol="features", numTrees=20)

# 4. Pipeline
pipeline_rf = Pipeline(stages=[assembler, rf])

# 5. Treino
model_rf = pipeline_rf.fit(train)

# 6. Avaliação
preds = model_rf.transform(test)
evaluator = BinaryClassificationEvaluator(labelCol="churn", metricName="areaUnderROC")
auc = evaluator.evaluate(preds)

print(f"Random Forest AUC: {auc:.4f}")
# (Passo 4: Pipeline com XGBoost (Avançado)) Nota: No Colab gratuito, instalar o wrapper Python do XGBoost4J pode ser complexo devido a versões de Scala. Vamos usar a API PySpark do XGBoost se disponível, ou simular com GradientBoostedTrees (GBT) nativo que é o “primo” do XGBoost. Para garantir que o lab funcione 100% no Colab sem erros de dependência Java, vamos usar o GBTClassifier do Spark, que usa a mesma lógica de Gradient Boosting.


Random Forest AUC: 0.9984


In [4]:
# --- Célula 4: Gradient Boosting (GBT) ---
from pyspark.ml.classification import GBTClassifier

# O GBT treina árvores sequencialmente para corrigir erros
gbt = GBTClassifier(labelCol="churn", featuresCol="features", maxIter=20)

pipeline_gbt = Pipeline(stages=[assembler, gbt])

print("Treinando Gradient Boosting...")
model_gbt = pipeline_gbt.fit(train)

preds_gbt = model_gbt.transform(test)
auc_gbt = evaluator.evaluate(preds_gbt)

print(f"Gradient Boosting AUC: {auc_gbt:.4f}")
# (Passo 5: Tuning com CrossValidator) Vamos achar o melhor maxDepth para o GBT.


Treinando Gradient Boosting...
Gradient Boosting AUC: 0.9995


In [6]:
# --- Célula 5: Hyperparameter Tuning ---
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Grid de Parâmetros
paramGrid = ParamGridBuilder() \
.addGrid(gbt.maxDepth, [2, 5]) \
.addGrid(gbt.stepSize, [0.1, 0.01]) \
.build()

# Cross Validator (3 Folds)
cv = CrossValidator(estimator=pipeline_gbt,
estimatorParamMaps=paramGrid,
evaluator=evaluator,
numFolds=3)

print("Iniciando Grid Search Distribuído (Isso pode demorar)...")
cvModel = cv.fit(train)

print(f"Melhor AUC encontrado: {max(cvModel.avgMetrics):.4f}")
print("Melhores parâmetros encontrados automaticamente!")
#(Análise do Lab) O CrossValidator usou todo o poder do cluster (no caso, os cores locais do Colab) para testar 4 combinações (2 depths * 2 steps) x 3 folds = 12 treinos em paralelo. Em um cluster real com 100 máquinas, isso seria instantâneo.
# Fim da Parte 2 da Etapa 2. Na próxima parte: Deep Learning Distribuído, TensorFlow on Spark e Laboratório de Classificação de Imagens.


Iniciando Grid Search Distribuído (Isso pode demorar)...
Melhor AUC encontrado: 0.9995
Melhores parâmetros encontrados automaticamente!


### Conclusão do Experimento

*   **Performance:** Ambos os modelos (Random Forest e GBT) atingiram AUC > 0.99, com leve vantagem para o Gradient Boosting.
*   **Escalabilidade:** O Spark MLlib permitiu treinar e otimizar (CrossValidation) os modelos de forma distribuída e eficiente.
*   **Tuning:** O Grid Search automatizou a busca pelos melhores hiperparâmetros, garantindo a melhor convergência do modelo de Churn.